# Alphabet 2026 위임장 본문 분리·청크 만들기
안건 목록을 기준으로 위임장 본문을 안건별로 잘라 정리하고, LLM 입력용 조각과 ESG 라벨 초안을 만든다.

출력 파일 (`data/processed/`):
- `googl_agenda_2026.csv` — 주주제안 제출자 열을 더해 덮어씀
- `googl_2026_sections/`, `googl_2026_sections_clean/` — 안건별 원본·정리 본문 txt
- `googl_2026_manifest.csv`, `googl_2026_chunks.csv` — 파일 목록과 LLM 입력 조각
- `googl_labels_2026.csv` — ESG 라벨 초안

## 1. 준비
HTML에서 화면에 보이는 본문만 꺼내고, 앞 노트북에서 만든 안건 CSV를 읽는다.

In [1]:
import re
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning

warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)
pd.set_option("display.max_colwidth", 150)

In [2]:
BASE = Path(r"C:\desktop\agent_contest")   # 프로젝트 폴더

HTML_PATH   = BASE / "data/raw/googl/def14a_2026.htm"
AGENDA_PATH = BASE / "data/processed/googl_agenda_2026.csv"
SECTION_DIR = BASE / "data/processed/googl_2026_sections"   # 다음 단계(본문 자르기) 저장 위치

for p in [HTML_PATH, AGENDA_PATH]:
    print(p.exists(), p)

True C:\desktop\agent_contest\data\raw\googl\def14a_2026.htm
True C:\desktop\agent_contest\data\processed\googl_agenda_2026.csv


In [3]:
html = HTML_PATH.read_text(encoding="utf-8", errors="ignore")
soup = BeautifulSoup(html, "lxml")

# 화면에 안 보이는 영역(iXBRL 숨김 정보) 제거
for h in soup.find_all(style=re.compile(r"display:\s*none", re.I)):
    h.decompose()

text = re.sub(r"\s+", " ", soup.body.get_text(" ", strip=True))
print(f"본문 글자 수: {len(text):,}")

본문 글자 수: 403,020


In [4]:
agenda = pd.read_csv(AGENDA_PATH)
print(agenda.shape)
agenda

(14, 6)


,proposal_no,title,proposer,board_rec,page,agenda_type
0,1,Election of Directors,회사,FOR,61,이사 선임
1,2,Ratification of Appointment of Independent Registered Public Accounting Firm,회사,FOR,62,회계법인 선임
2,3,Approval of the Amendment and Restatement of Alphabet Inc. Amended and Restated 2021 Stock Plan,회사,FOR,63,주식보상계획
3,4,Advisory Vote to Approve Compensation Awarded to Named Executive Officers,회사,FOR,68,경영진 보수
4,5,Shareholder Proposal Regarding an Enhanced Disclosure on Climate Goals,주주,AGAINST,72,주주제안
5,6,Shareholder Proposal Regarding a Report on Water Usage and AI Development,주주,AGAINST,74,주주제안
6,7,Shareholder Proposal Regarding Equal Shareholder Voting,주주,AGAINST,77,주주제안
7,8,Shareholder Proposal Regarding a Viewpoint Diversity Risk Report,주주,AGAINST,80,주주제안
8,9,Shareholder Proposal Regarding a Report on Politicized Content Moderation,주주,AGAINST,83,주주제안
9,10,Shareholder Proposal Regarding a Report on Impact of U.S. Immigration Policy,주주,AGAINST,85,주주제안


## 2. 안건별 본문 자르기
제목이 마지막으로 나온 위치를 시작점으로 자르고(14번 끝은 Q&A 제목), `googl_2026_sections/`에 저장한다.

In [5]:
def title_regex(title, n_chars=50):
    """제목 앞 50자를, 글자 사이에 공백이 끼어 있어도 찾을 수 있는 패턴으로 바꿈"""
    key = title.replace(" ", "")[:n_chars]
    return re.compile(r"\s?".join(re.escape(ch) for ch in key))

starts, counts = [], []
for title in agenda["title"]:
    found = [m.start() for m in title_regex(title).finditer(text)]
    counts.append(len(found))
    starts.append(found[-1] if found else None)   # 마지막으로 나온 곳 = 본문 시작

agenda["start"] = starts
agenda["occurrences"] = counts

print("시작 위치가 안건 순서대로 커지는가:", agenda["start"].is_monotonic_increasing)
agenda[["proposal_no", "title", "occurrences", "start"]]

시작 위치가 안건 순서대로 커지는가: True


,proposal_no,title,occurrences,start
0,1,Election of Directors,2,194358
1,2,Ratification of Appointment of Independent Registered Public Accounting Firm,2,197362
2,3,Approval of the Amendment and Restatement of Alphabet Inc. Amended and Restated 2021 Stock Plan,2,199716
3,4,Advisory Vote to Approve Compensation Awarded to Named Executive Officers,2,222201
4,5,Shareholder Proposal Regarding an Enhanced Disclosure on Climate Goals,2,232822
5,6,Shareholder Proposal Regarding a Report on Water Usage and AI Development,2,241235
6,7,Shareholder Proposal Regarding Equal Shareholder Voting,2,250366
7,8,Shareholder Proposal Regarding a Viewpoint Diversity Risk Report,2,259414
8,9,Shareholder Proposal Regarding a Report on Politicized Content Moderation,2,269709
9,10,Shareholder Proposal Regarding a Report on Impact of U.S. Immigration Policy,2,278184


In [6]:
# 14번 뒤에는 다음 안건이 없으니, 그 뒤에 나오는 큰 섹션 제목을 끝으로 씀
end_markers = ["Questions and Answers", "Q&A", "Information About the Meeting",
               "Information about the Meeting", "Appendix"]

last_start = agenda["start"].iloc[-1]
marker_pos = {mk: text.find(mk, last_start + 200) for mk in end_markers}
marker_pos = {mk: pos for mk, pos in marker_pos.items() if pos != -1}

last_end_marker = min(marker_pos, key=marker_pos.get) if marker_pos else None
last_end = marker_pos[last_end_marker] if last_end_marker else len(text)

print("찾은 끝 표시들:", marker_pos)
print("14번 끝 기준:", last_end_marker, "/ 위치:", f"{last_end:,}")
print("14번 길이:", f"{last_end - last_start:,}자")

찾은 끝 표시들: {'Questions and Answers': 329732, 'Q&A': 322460, 'Appendix': 322464}
14번 끝 기준: Q&A / 위치: 322,460
14번 길이: 4,177자


In [7]:
for name, pos in [("Q&A", 322460), ("Questions and Answers", 329732)]:
    print(f"===== {name} 주변 =====")
    print(text[pos-300 : pos+300])
    print()

===== Q&A 주변 =====
edu/news/privacy-ai-era-how-do-we-protect-our-personal-information (6) https://www.skadden.com/insights/publications/2025/05/copyright-office-report 96 Alphabet 2026 Proxy Statement Proxy Statement Summary & Highlights Corporate Governance Director and Executive Compensation Audit Matters Proposals Q&A Appendix (7) https://www.forbes.com/councils/forbestechcouncil/2025/11/13/the-ai-security-blindspot-how-unauthorized-ai-tools-are-creating-corporate-vulnerabilities/ (8) https://www.fool.com/investing/2025/08/16/this-underrated-artificial-intelligence-ai-stock-h/ (9) https://companiesmarketcap.c

===== Questions and Answers 주변 =====
d AGAINST the shareholder proposal. Alphabet Recommendation OUR BOARD OF DIRECTORS RECOMMENDS A VOTE AGAINST THE SHAREHOLDER PROPOSAL. 98 Alphabet 2026 Proxy Statement Proxy Statement Summary & Highlights Corporate Governance Director and Executive Compensation Audit Matters Proposals Q&A Appen dix Questions and Answers About the Proxy Mate

In [8]:
# 1) 14번 끝 = 실제 Q&A 섹션 제목
last_end = text.find("Questions and Answers About the Proxy Materials", last_start)
print("14번 끝 위치:", f"{last_end:,}", "/ 14번 길이:", f"{last_end - last_start:,}자\n")

# 2) N번 시작 ~ (N+1)번 시작 전까지 자르기
agenda["end"] = list(agenda["start"].iloc[1:]) + [last_end]
agenda["section_text"] = [text[s:e] for s, e in zip(agenda["start"], agenda["end"])]
agenda["length"] = agenda["section_text"].str.len()

# 3) 안건별 txt 파일로 저장
SECTION_DIR.mkdir(parents=True, exist_ok=True)
for _, r in agenda.iterrows():
    (SECTION_DIR / f"proposal_{r['proposal_no']:02d}.txt").write_text(r["section_text"], encoding="utf-8")
print("저장 폴더:", SECTION_DIR, "\n")

# 4) 확인용: 길이와 앞부분 80자
agenda["preview"] = agenda["section_text"].str[:80]
agenda[["proposal_no", "length", "preview"]]

14번 끝 위치: 329,732 / 14번 길이: 11,449자

저장 폴더: C:\desktop\agent_contest\data\processed\googl_2026_sections 



,proposal_no,length,preview
0,1,3004,"Election of Directors Nominees The Governance Committee recommended, and our Boa"
1,2,2354,Ratification of Appointment of Independent Registered Public Accounting Firm The
2,3,22485,Approval of the Amendment and Restatement of Alphabet Inc. Amended and Restated
3,4,10621,Advisory Vote to Approve Compensation Awarded to Named Executive Officers As req
4,5,8413,Shareholder Proposal Regarding an Enhanced Disclosure on Climate Goals Trillium
5,6,9131,Shareholder Proposal Regarding a Report on Water Usage and AI Development Inspir
6,7,9048,Shareholder Proposal Regarding Equal Shareholder Voting The NorthStar Asset Mana
7,8,10295,Shareholder Proposal Regarding a Viewpoint Diversity Risk Report The National Ce
8,9,8475,Shareholder Proposal Regarding a Report on Politicized Content Moderation Bowyer
9,10,7468,Shareholder Proposal Regarding a Report on Impact of U.S. Immigration Policy The


## 3. 머리말·메뉴 제거
쪽마다 반복되는 'Alphabet 2026 Proxy Statement'·쪽번호·메뉴를 지워 `googl_2026_sections_clean/`에 저장한다.

In [9]:
# 페이지 메뉴의 시작점: "Alphabet 2026 Proxy Statement"
anchor = re.compile(r"Alphabet\s?2026\s?Proxy\s?Statement")

# (1) 위임장 전체에서 몇 번 나오는지
hits = [m.start() for m in anchor.finditer(text)]
print("위임장 전체 등장 횟수:", len(hits), "\n")

# (2) 앞 10자 + 뒤 140자를 잘라서, 같은 모양끼리 묶어 세기
shapes = Counter(text[max(0, h - 10): h + 140] for h in hits)
print("서로 다른 모양 개수:", len(shapes), "\n")
for shape, cnt in shapes.most_common(5):
    print(f"[{cnt}회] {shape}\n")

# (3) 안건별 본문 안에 몇 번 섞여 있는지
agenda["menu_hits"] = agenda["section_text"].apply(lambda s: len(anchor.findall(s)))
agenda[["proposal_no", "length", "menu_hits"]]

위임장 전체 등장 횟수: 113 

서로 다른 모양 개수: 113 

[1회] 4, 2026 2 Alphabet 2026 Proxy Statement Letter from the Chair of the Board of Directors Dear Fellow Shareholders, The past year has been one of substa

[1회] Directors Alphabet 2026 Proxy Statement 3 Notice of 2026 Annual Meeting of Shareholders Date & Time Friday, June 5, 2026 9:00 a.m., Pacific Time Virtu

[1회] phabet. 4 Alphabet 2026 Proxy Statement Important Notice Regarding Internet Availability of Proxy Materials This proxy statement and our 2025 Annual R

[1회] ces only. Alphabet 2026 Proxy Statement 5 Proxy Statement Summary & Highlights Corporate Governance Director and Executive Compensation Audit Matters 

[1회] served. 6 Alphabet 2026 Proxy Statement Proxy Statement Summary & Highlights Corporate Governance Director and Executive Compensation Audit Matters Pr



,proposal_no,length,menu_hits
0,1,3004,1
1,2,2354,1
2,3,22485,5
3,4,10621,4
4,5,8413,2
5,6,9131,3
6,7,9048,3
7,8,10295,3
8,9,8475,2
9,10,7468,2


In [10]:
MENU = ("Proxy Statement Summary & Highlights Corporate Governance "
        "Director and Executive Compensation Audit Matters Proposals Q&A Appendix")

def loose(s):
    """글자 사이 공백이 있어도/없어도 잡히는 패턴"""
    return r"\s?".join(re.escape(ch) for ch in s.replace(" ", ""))

header_pat = re.compile(
    r"(?:\b\d{1,3}\s)?"                              # 앞 쪽번호 (짝수 쪽)
    r"Alphabet\s?2026\s?Proxy\s?Statement"
    r"(?:\s\d{1,3}\b)?"                              # 뒤 쪽번호 (홀수 쪽)
    r"(?:\s?" + loose(MENU) + r")?"                  # 메뉴 (있을 때만)
)

found = [m.group() for m in header_pat.finditer(text)]
with_menu = [f for f in found if "Appen" in f]
print("잡힌 개수:", len(found), "/ 그중 메뉴 포함:", len(with_menu), "\n")

# 시험 삭제 후, 메뉴 흔적이 남아 있는지 확인
test = header_pat.sub(" ", text)
left = re.findall(r"Q\s?&\s?A\s?Appen\s?dix", test)
print("지운 뒤 남은 메뉴 흔적:", len(left), "\n")

# 잡힌 예시 3개
for f in found[3:6]:
    print("▶", f, "\n")

잡힌 개수: 113 / 그중 메뉴 포함: 106 

지운 뒤 남은 메뉴 흔적: 0 

▶ Alphabet 2026 Proxy Statement 5 Proxy Statement Summary & Highlights Corporate Governance Director and Executive Compensation Audit Matters Proposals Q&A Appendix 

▶ 6 Alphabet 2026 Proxy Statement Proxy Statement Summary & Highlights Corporate Governance Director and Executive Compensation Audit Matters Proposals Q&A Appendix 

▶ Alphabet 2026 Proxy Statement 7 Proxy Statement Summary & Highlights Corporate Governance Director and Executive Compensation Audit Matters Proposals Q&A Appendix 



In [11]:
CLEAN_DIR = BASE / "data/processed/googl_2026_sections_clean"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

def clean_section(s):
    s = header_pat.sub(" ", s)          # 머리말·꼬리말·메뉴 지우기
    return re.sub(r"\s+", " ", s).strip()  # 지운 자리의 연속 공백 정리

agenda["removed"] = agenda["section_text"].apply(lambda s: len(header_pat.findall(s)))
agenda["clean_text"] = agenda["section_text"].apply(clean_section)
agenda["clean_length"] = agenda["clean_text"].str.len()

for _, r in agenda.iterrows():
    (CLEAN_DIR / f"proposal_{r['proposal_no']:02d}.txt").write_text(r["clean_text"], encoding="utf-8")

print("저장 폴더:", CLEAN_DIR)
print("정리 후 남은 'Alphabet 2026 Proxy Statement':",
      agenda["clean_text"].str.count(r"Alphabet\s?2026\s?Proxy\s?Statement").sum(), "\n")

agenda[["proposal_no", "length", "clean_length", "removed"]]

저장 폴더: C:\desktop\agent_contest\data\processed\googl_2026_sections_clean
정리 후 남은 'Alphabet 2026 Proxy Statement': 0 



,proposal_no,length,clean_length,removed
0,1,3004,2839,1
1,2,2354,2189,1
2,3,22485,21664,5
3,4,10621,9964,4
4,5,8413,8084,2
5,6,9131,8638,3
6,7,9048,8555,3
7,8,10295,9802,3
8,9,8475,8146,2
9,10,7468,7138,2


## 4. 주주제안 제출자 추출
'has/have advised us' 앞부분에서 제출 기관·실제 주주·공동 제안자를 뽑아 안건 CSV에 더한다.

In [12]:
holder = agenda[agenda["proposer"] == "주주"]

for _, r in holder.iterrows():
    body = r["clean_text"][len(r["title"]):].strip()   # 제목 부분은 빼고
    print(f"[{r['proposal_no']}] {body[:300]}\n")

[5] Trillium ESG Global Equity Fund, as lead filer, along with other co-filers, whose names, addresses, and shareholdings will be provided by us upon request, have advised us that they intend to submit the following proposal for consideration at our Annual Meeting. Enhanced Disclosure Whereas: Climate c

[6] Inspire Investing, LLC, on behalf of its client, has advised us that it intends to submit the following proposal for consideration at our Annual Meeting. Report Regarding Water Usage and AI Development Whereas: Alphabet’s leadership in AI and cloud computing depends on reliable access to water resou

[7] The NorthStar Asset Management, Inc. Funded Pension Plan, as lead filer, and Mercy Investment Services, Inc., as co-filer, have advised us that they intend to submit the following proposal for consideration at our Annual Meeting. Give Each Share an Equal Vote RESOLVED: Shareholders request that our 

[8] The National Center for Public Policy Research has advised us that it intends 

In [13]:
def parse_filer(row):
    body = row["clean_text"][len(row["title"]):].strip()

    # 1) "has/have advised us" 앞부분만 꺼내기
    m = re.search(r"\s(?:has|have) advised us", body[:800])
    if not m:
        return pd.Series({"filer": None, "on_behalf_of": None, "co_filers": None})
    head = body[:m.start()].strip().rstrip(",")

    # 2) 공동 제안자 확인
    named = re.search(r"as lead filer,\s*and (.+?),\s*as co-filer", head)
    if named:
        co = named.group(1)                 # 이름이 나와 있는 경우 (7번)
    elif "co-filer" in head:
        co = "있음(이름 미공개)"             # "요청 시 제공" 경우
    else:
        co = "없음"

    # 3) ", as lead filer ..." 뒷부분 버리기
    main = re.split(r",\s*as lead filer", head)[0]

    # 4) "on behalf of" 앞 = 제출 기관, 뒤 = 실제 주주
    if ", on behalf of " in main:
        filer, behalf = main.split(", on behalf of ", 1)
    else:
        filer, behalf = main, None

    return pd.Series({"filer": filer.strip(), "on_behalf_of": behalf, "co_filers": co})


holder_mask = agenda["proposer"] == "주주"
agenda.loc[holder_mask, ["filer", "on_behalf_of", "co_filers"]] = \
    agenda[holder_mask].apply(parse_filer, axis=1)

agenda.loc[holder_mask, ["proposal_no", "filer", "on_behalf_of", "co_filers"]]

,proposal_no,filer,on_behalf_of,co_filers
4,5,Trillium ESG Global Equity Fund,NaN,있음(이름 미공개)
5,6,"Inspire Investing, LLC",its client,없음
6,7,"The NorthStar Asset Management, Inc. Funded Pension Plan",NaN,"Mercy Investment Services, Inc."
7,8,The National Center for Public Policy Research,NaN,없음
8,9,"Bowyer Research, Inc.",The Heritage Foundation,없음
9,10,The SOC Investment Group,NaN,없음
10,11,Zevin Asset Management,Denise and Gerald Bergman,있음(이름 미공개)
11,12,The Shareholder Association for Research & Education,the Pension Plan of The United Church of Canada,있음(이름 미공개)
12,13,Vancity Investment Management,NaN,없음
13,14,The National Legal and Policy Center,NaN,없음


In [14]:
cols = ["proposal_no", "title", "proposer", "board_rec", "page", "agenda_type",
        "filer", "on_behalf_of", "co_filers"]

agenda[cols].to_csv(AGENDA_PATH, index=False, encoding="utf-8-sig")
print("저장 완료:", AGENDA_PATH, "\n")

pd.read_csv(AGENDA_PATH)   # 다시 읽어서 확인

저장 완료: C:\desktop\agent_contest\data\processed\googl_agenda_2026.csv 



,proposal_no,title,proposer,board_rec,page,agenda_type,filer,on_behalf_of,co_filers
0,1,Election of Directors,회사,FOR,61,이사 선임,NaN,NaN,NaN
1,2,Ratification of Appointment of Independent Registered Public Accounting Firm,회사,FOR,62,회계법인 선임,NaN,NaN,NaN
2,3,Approval of the Amendment and Restatement of Alphabet Inc. Amended and Restated 2021 Stock Plan,회사,FOR,63,주식보상계획,NaN,NaN,NaN
3,4,Advisory Vote to Approve Compensation Awarded to Named Executive Officers,회사,FOR,68,경영진 보수,NaN,NaN,NaN
4,5,Shareholder Proposal Regarding an Enhanced Disclosure on Climate Goals,주주,AGAINST,72,주주제안,Trillium ESG Global Equity Fund,NaN,있음(이름 미공개)
5,6,Shareholder Proposal Regarding a Report on Water Usage and AI Development,주주,AGAINST,74,주주제안,"Inspire Investing, LLC",its client,없음
6,7,Shareholder Proposal Regarding Equal Shareholder Voting,주주,AGAINST,77,주주제안,"The NorthStar Asset Management, Inc. Funded Pension Plan",NaN,"Mercy Investment Services, Inc."
7,8,Shareholder Proposal Regarding a Viewpoint Diversity Risk Report,주주,AGAINST,80,주주제안,The National Center for Public Policy Research,NaN,없음
8,9,Shareholder Proposal Regarding a Report on Politicized Content Moderation,주주,AGAINST,83,주주제안,"Bowyer Research, Inc.",The Heritage Foundation,없음
9,10,Shareholder Proposal Regarding a Report on Impact of U.S. Immigration Policy,주주,AGAINST,85,주주제안,The SOC Investment Group,NaN,없음


## 5. 1번 보충: 이사 소개·선임 절차
두 섹션을 위치로 잘라 `proposal_01_supp_directors.txt`, `proposal_01_supp_selection.txt`로 저장한다.

In [15]:
p1 = agenda.loc[agenda["proposal_no"] == 1, "clean_text"].iloc[0]
print(len(p1), "자\n")
print(p1)

2839 자

Election of Directors Nominees The Governance Committee recommended, and our Board nominated: • Larry Page • Sergey Brin • Sundar Pichai • John L. Hennessy • Frances H. Arnold • R. Martin “Marty” Chávez • L. John Doerr • Roger W. Ferguson Jr. • K. Ram Shriram • Robin L. Washington as nominees for election as members of our Board at the Annual Meeting. At the Annual Meeting, ten directors will be elected to our Board. Except as set forth below, unless otherwise instructed, the persons appointed in the accompanying form of proxy will vote the proxies received by them for these nominees, who are all presently directors of Alphabet. In the event that any nominee becomes unavailable or unwilling to serve as a member of our Board, the proxy holders will vote in their discretion for a substitute nominee. The term of office of each person elected as a director will continue until the next annual meeting or until a successor has been elected and qualified, or until the director’s earlie

In [16]:
targets = ["Directors and Executive Officers",
           "Director Selection Process and Qualifications"]

for t in targets:
    hits = [m.start() for m in title_regex(t).finditer(text)]
    print(f"===== {t} : {len(hits)}회 =====")
    for h in hits:
        print(f"[{h:,}] ...{text[h-60:h+140]}...\n")

===== Directors and Executive Officers : 3회 =====
[36,295] ... Directors, Executive Officers, and Corporate Governance 16 Directors and Executive Officers 16 Corporate Governance and Board Matters 24 Board Meetings 24 Board Leadership Structure 24 Board Committe...

[40,272] ...dix Directors, Executive Officers, and Corporate Governance Directors and Executive Officers Our Board of Directors (our Board) is composed of highly experienced leaders who have shaped prominent glob...

[195,405] ...arlier death, resignation, or removal. The sections titled “Directors and Executive Officers” and “Director Selection Process and Qualifications” on pages 16 and 31 of this proxy statement contain mor...

===== Director Selection Process and Qualifications : 3회 =====
[82,339] ...ard and to address the membership criteria set forth under “Director Selection Process and Qualifications” on page 31 of this proxy statement. Any shareholder recommendations for consideration by the ...

[84,389] ...Execut

In [17]:
print(text[36000:37800])

risks • We actively collaborate with the broader online ecosystem to advance privacy standards and mitigate security risks Alphabet 2026 Proxy Statement 13 Table of Contents Proxy Statement Summary & Highlights 6 1 Corporate Governance Directors, Executive Officers, and Corporate Governance 16 Directors and Executive Officers 16 Corporate Governance and Board Matters 24 Board Meetings 24 Board Leadership Structure 24 Board Committees 24 Director Independence 30 Compensation Committee Interlocks and Insider Participation 30 Consideration of Director Nominees 30 Director Service on Outside Boards and Other Commitments 31 Management Succession Planning 32 Board’s Role in Risk Oversight 32 Executive Sessions 33 Outside Advisors 33 Board Effectiveness, Board Annual Self-Assessment, Board Education 33 Sha reholder Engagement 34 Communications with our Board 34 Common Stock Ownership of Certain Beneficial Owners and Management 35 Delinquent Section 16(a) Reports 37 Certain Relationships and R

In [18]:
supp = {
    "directors": (40272, "Corporate Governance and Board Matters"),
    "selection": (84389, "Director Service on Outside Boards"),
}

for name, (start, end_title) in supp.items():
    m = title_regex(end_title).search(text, start + 100)   # 시작 이후 처음 나오는 다음 제목
    end = m.start()
    piece = clean_section(text[start:end])                 # 셀 10의 머리말 정리 함수 재사용

    out = CLEAN_DIR / f"proposal_01_supp_{name}.txt"
    out.write_text(piece, encoding="utf-8")

    print(f"===== {name} : {start:,} ~ {end:,} / 정리 후 {len(piece):,}자 =====")
    print("앞:", piece[:150])
    print("끝:", piece[-150:], "\n")

===== directors : 40,272 ~ 64,813 / 정리 후 23,228자 =====
앞: Directors and Executive Officers Our Board of Directors (our Board) is composed of highly experienced leaders who have shaped prominent global organiz
끝: at various technology companies, and providing in-depth knowledge of the technology sector. • Currently co-chairs Google’s Trust & Compliance Council. 

===== selection : 84,389 ~ 87,581 / 정리 후 3,191자 =====
앞: Director Selection Process and Qualifications The Governance Committee will evaluate and recommend candidates for membership on our Board consistent w
끝:  biographies on pages 16 - 21 provide us with a wide range of perspectives and judgment necessary to guide our strategies and monitor their execution. 



## 6. 주주제안을 주주 요구와 회사 반론으로 나누기
'Alphabet Opposing Statement'를 경계로 `_holder.txt`, `_company.txt` 파일을 만든다.

In [19]:
candidates_kw = [
    "Statement in Opposition", "Board's Statement", "Board’s Statement",
    "Our Board's Response", "Our Board’s Response", "Alphabet's Response",
    "Alphabet’s Response", "Response to", "Board Recommendation",
    "Alphabet Recommendation", "Our Board", "We believe",
]

holder = agenda[agenda["proposer"] == "주주"]
rows = []
for _, r in holder.iterrows():
    s = r["clean_text"]
    row = {"proposal_no": r["proposal_no"], "length": len(s)}
    for kw in candidates_kw:
        pos = s.find(kw)
        row[kw] = pos if pos != -1 else None   # 처음 나온 위치
    rows.append(row)

kw_table = pd.DataFrame(rows).set_index("proposal_no")
kw_table.dropna(axis=1, how="all")   # 한 번도 안 나온 표현은 숨김

,length,Alphabet Recommendation,Our Board,We believe
proposal_no,,,,
5,8084,7967,4480,6610.0
6,8638,8521,4326,7640.0
7,8555,8438,3617,NaN
8,9802,9685,4437,NaN
9,8146,8028,5010,NaN
10,7138,7020,4113,NaN
11,10218,10100,4214,9184.0
12,9943,9825,3852,NaN
13,10991,10873,5862,NaN


In [20]:
for no, r in holder.set_index("proposal_no").iterrows():
    s = r["clean_text"]
    pos = s.find("Our Board")
    print(f"[{no}] ...{s[pos-150:pos+150]}...\n")

[5] ...mendations-for-policy-makers-on-net-zero-action-aligning-commitments-with-science-based-no-low-overshoot-1-5-c-scenarios/ Alphabet Opposing Statement Our Board of Directors recommends a vote AGAINST the shareholder proposal because: • Our existing environmental reporting has long provided detailed a...

[6] ...live.com/article/apple-leads-us-dominates-kantars-top-100-global-brands/1918351 (5) https://finance.yahoo.com/quote/GOOG/ Alphabet Opposing Statement Our Board of Directors recommends a vote AGAINST the shareholder proposal because: • We have a long track record of providing extensive information an...

[7] ...s supported the proposal. See Alphabet Inc., Form 8-K (June 12, 2025); DEF 14A (Apr. 25, 2025); Form 10-K (Feb. 4, 2025). Alphabet Opposing Statement Our Board of Directors recommends a vote AGAINST the shareholder proposal because: • Our current capital structure has provided significant long-term ...

[8] ...ww.foxbusiness.com/media/robby-starbuck-why-he-sued-googl

In [21]:
SPLIT = "Alphabet Opposing Statement"
FINAL = "Alphabet Recommendation"

for idx, r in holder.iterrows():
    s = r["clean_text"]
    assert s.count(SPLIT) == 1, f"{r['proposal_no']}번: 경계 제목이 1번이 아님"

    b = s.find(SPLIT)
    f = s.rfind(FINAL)
    holder_part  = s[len(r["title"]):b].strip()      # 제목 뒤 ~ 경계 전
    company_part = s[b + len(SPLIT):f].strip()       # 경계 뒤 ~ 최종 권고 전

    no = r["proposal_no"]
    (CLEAN_DIR / f"proposal_{no:02d}_holder.txt").write_text(holder_part, encoding="utf-8")
    (CLEAN_DIR / f"proposal_{no:02d}_company.txt").write_text(company_part, encoding="utf-8")

    agenda.loc[idx, "holder_len"] = len(holder_part)
    agenda.loc[idx, "company_len"] = len(company_part)

agenda.loc[agenda["proposer"] == "주주", ["proposal_no", "holder_len", "company_len"]]

,proposal_no,holder_len,company_len
4,5,4380.0,3486.0
5,6,4223.0,4194.0
6,7,3532.0,4820.0
7,8,4343.0,5247.0
8,9,4907.0,3017.0
9,10,4007.0,2906.0
10,11,4129.0,5885.0
11,12,3773.0,5972.0
12,13,5762.0,5010.0
13,14,6090.0,4670.0


## 7. 1번 보충: 이사회 리더십·독립성·겸직
지배구조 섹션 3개를 `proposal_01_supp_*.txt`로 추가 저장한다.

In [22]:
g_targets = ["Board Leadership Structure",
             "Director Independence",
             "Director Service on Outside Boards"]

for t in g_targets:
    hits = [m.start() for m in title_regex(t).finditer(text)]
    print(f"===== {t} : {len(hits)}회 =====")
    for h in hits:
        print(f"[{h:,}] ...{text[h-40:h+120]}...\n")

===== Board Leadership Structure : 2회 =====
[36,391] ... and Board Matters 24 Board Meetings 24 Board Leadership Structure 24 Board Committees 24 Director Independence 30 Compensation Committee Interlocks and Insider...

[66,284] ...ur 2025 Annual Meeting of Shareholders. Board Leadership Structure In January 2018, John L. Hennessy, the then Lead Independent Director, was appointed to serve...

===== Director Independence : 3회 =====
[36,441] ...ership Structure 24 Board Committees 24 Director Independence 30 Compensation Committee Interlocks and Insider Participation 30 Consideration of Director Nomine...

[67,540] ...ergey, and Sundar, is independent (see “Director Independence” on page 30 of this proxy statement), and our Board believes that the independent directors provid...

[80,482] ...on Audit Matters Proposals Q&A Appendix Director Independence Our Board has adopted independence standards that mirror the criteria specified by applicable laws...

===== Director Service on Outsid

In [23]:
g_supp = {
    "leadership":   (66284, "Board Committees"),
    "independence": (80482, "Compensation Committee Interlocks"),
    "outside_boards": (87581, "Management Succession Planning"),
}

for name, (start, end_title) in g_supp.items():
    m = title_regex(end_title).search(text, start + 100)
    end = m.start()
    piece = clean_section(text[start:end])

    out = CLEAN_DIR / f"proposal_01_supp_{name}.txt"
    out.write_text(piece, encoding="utf-8")

    print(f"===== {name} : {start:,} ~ {end:,} / 정리 후 {len(piece):,}자 =====")
    print("앞:", piece[:150])
    print("끝:", piece[-150:], "\n")

===== leadership : 66,284 ~ 67,697 / 정리 후 1,412자 =====
앞: Board Leadership Structure In January 2018, John L. Hennessy, the then Lead Independent Director, was appointed to serve as Alphabet’s Chair of the Bo
끝: or Independence” on page 30 of this proxy statement), and our Board believes that the independent directors provide effective oversight of management. 

===== independence : 80,482 ~ 81,087 / 정리 후 604자 =====
앞: Director Independence Our Board has adopted independence standards that mirror the criteria specified by applicable laws and regulations of the SEC an
끝:  director had any interest, including those discussed under “Certain Relationships and Related Transactions” on pages 38 - 39 of this proxy statement. 

===== outside_boards : 87,581 ~ 88,403 / 정리 후 657자 =====
앞: Director Service on Outside Boards and Other Commitments Each member of our Board is expected to ensure that other existing and future commitments, in
끝: ompany boards our directors can serve on is four, includ

## 8. 본문 끝 정리와 주주제안 공통 섹션 분리
끝에 붙은 'Proposal Number N' 라벨을 지우고, 4번 뒤에 붙은 주주제안 안내를 `shareholder_proposals_overview.txt`로 뗀다.

In [24]:
label_pat = re.compile(r"\s*Proposal\s?Number\s?\d{1,2}\s?:?\s*$")

agenda["had_label"] = agenda["clean_text"].str.contains(label_pat)
agenda["clean_text"] = agenda["clean_text"].str.replace(label_pat, "", regex=True)

# 정리본 다시 저장 (전체 본문 파일만, holder/company 파일은 해당 없음)
for _, r in agenda.iterrows():
    (CLEAN_DIR / f"proposal_{r['proposal_no']:02d}.txt").write_text(r["clean_text"], encoding="utf-8")

agenda["tail"] = agenda["clean_text"].str[-60:]
agenda[["proposal_no", "had_label", "tail"]]

,proposal_no,had_label,tail
0,1,True,E BOARD OF DIRECTORS OF EACH OF THE ABOVEMENTIONED NOMINEES.
1,2,True,"CCOUNTING FIRM FOR THE FISCAL YEAR ENDING DECEMBER 31, 2026."
2,3,True,"CAPITAL STOCK ISSUABLE UNDER THE PLAN BY 200,000,000 SHARES."
3,4,True,043 Email: corporatesecretary@abc.xyz Number: (650) 253-3393
4,5,True,IRECTORS RECOMMENDS A VOTE AGAINST THE SHAREHOLDER PROPOSAL.
5,6,True,IRECTORS RECOMMENDS A VOTE AGAINST THE SHAREHOLDER PROPOSAL.
6,7,True,IRECTORS RECOMMENDS A VOTE AGAINST THE SHAREHOLDER PROPOSAL.
7,8,True,IRECTORS RECOMMENDS A VOTE AGAINST THE SHAREHOLDER PROPOSAL.
8,9,True,IRECTORS RECOMMENDS A VOTE AGAINST THE SHAREHOLDER PROPOSAL.
9,10,True,IRECTORS RECOMMENDS A VOTE AGAINST THE SHAREHOLDER PROPOSAL.


In [25]:
p4 = agenda.loc[agenda["proposal_no"] == 4, "clean_text"].iloc[0]

rec_pos = p4.rfind("RECOMMENDS A VOTE")
print("4번 길이:", len(p4), "/ 마지막 권고 문장 위치:", rec_pos, "/ 그 뒤 글자 수:", len(p4) - rec_pos, "\n")
print(p4[rec_pos - 100:])

4번 길이: 9945 / 마지막 권고 문장 위치: 2005 / 그 뒤 글자 수: 7940 

valuating Alphabet’s executive compensation program. Alphabet Recommendation OUR BOARD OF DIRECTORS RECOMMENDS A VOTE FOR THE APPROVAL, ON AN ADVISORY BASIS, OF THE COMPENSATION AWARDED TO ALPHABET’S NAMED EXECUTIVE OFFICERS AS DESCRIBED IN THIS PROXY STATEMENT. Shareholder Proposals Our Approach We are committed to advancing our practices, policies, and disclosures in ways that further the interests of the company and ultimately drive long-term shareholder value. We recognize that the submission of proposals for vote at the Annual Meeting is one mechanism for our shareholders to convey their priorities and perspectives. Our Board and management team evaluate each shareholder proposal through a framework that considers the following factors, among others: • Existing initiatives and disclosures. Shareholder proposals often request that we prepare a report, adopt a policy, or implement new (or different) processes. We appreciate the cor

In [26]:
SP_HEAD = "Shareholder Proposals Our Approach"

p4 = agenda.loc[agenda["proposal_no"] == 4, "clean_text"].iloc[0]
cut = p4.find(SP_HEAD)
print("분리 위치:", cut, "\n")

p4_only  = p4[:cut].strip()
overview = p4[cut:].strip()

# 4번 본문 갱신 + 저장
agenda.loc[agenda["proposal_no"] == 4, "clean_text"] = p4_only
(CLEAN_DIR / "proposal_04.txt").write_text(p4_only, encoding="utf-8")

# 공통 섹션 따로 저장
(CLEAN_DIR / "shareholder_proposals_overview.txt").write_text(overview, encoding="utf-8")

print(f"4번 본문: {len(p4_only):,}자 / 끝: ...{p4_only[-80:]}\n")
print(f"공통 섹션: {len(overview):,}자 / 앞: {overview[:80]}...")

분리 위치: 2168 

4번 본문: 2,167자 / 끝: ...DED TO ALPHABET’S NAMED EXECUTIVE OFFICERS AS DESCRIBED IN THIS PROXY STATEMENT.

공통 섹션: 7,777자 / 앞: Shareholder Proposals Our Approach We are committed to advancing our practices, ...


## 9. 4번 보충: CD&A
보수 설명(CD&A) 범위를 확인해 `proposal_04_supp_cda.txt`로 저장한다.

In [27]:
cda_targets = ["Compensation Discussion and Analysis",
               "Leadership Development, Inclusion and Compensation Committee Report"]

for t in cda_targets:
    hits = [m.start() for m in title_regex(t).finditer(text)]
    print(f"===== {t} : {len(hits)}회 =====")
    for h in hits:
        print(f"[{h:,}] ...{text[h-40:h+120]}...\n")

===== Compensation Discussion and Analysis : 9회 =====
[26,199] ...ensation philosophy and practices, see “Compensation Discussion and Analysis” beginning on page 42 . 10 Alphabet 2026 Proxy Statement Proxy Statement Summary & ...

[37,290] ... for 202 5 41 Executive Compensation 42 Compensation Discussion and Analysis 42 Section 1 — Executive Summary 42 Section 2 — Determining Competitive Levels of P...

[76,592] ... • Reviewing with management our annual Compensation Discussion and Analysis (CD&A). • Preparing and approving the annual Compensation Committee Report. During ...

[116,805] ...panies (identified in Section 2 of the “Compensation Discussion and Analysis”) that compete with us for talent. We designed the program to address the time, eff...

[122,953] ...ion 5—Other Compensation Information 47 Compensation Discussion and Analysis Our Compensation Discussion and Analysis (CD&A) includes a detailed discussion of c...

[122,994] ...ompensation Discussion and Analysis Our Compen

In [28]:
end_t = "Leadership Development, Inclusion and Compensation Committee Report"

hits = [m.start() for m in title_regex(end_t).finditer(text, 122953, 151100)]
print(f"범위 안 등장: {len(hits)}회\n")
for h in hits:
    print(f"[{h:,}] ...{text[h-60:h+160]}...\n")

범위 안 등장: 2회

[150,910] ...e pension plans or supplemental executive retirement plans. Leadership Development, Inclusion and Compensation Committee Report The Leadership Development, Inclusion and Compensation Committee has reviewed and discussed ...

[150,982] ...evelopment, Inclusion and Compensation Committee Report The Leadership Development, Inclusion and Compensation Committee has reviewed and discussed the Compensation Discussion and Analysis with management. Based on its r...



In [29]:
start, end = 122953, 150910
piece = clean_section(text[start:end])

out = CLEAN_DIR / "proposal_04_supp_cda.txt"
out.write_text(piece, encoding="utf-8")

print(f"CD&A : {start:,} ~ {end:,} / 정리 후 {len(piece):,}자")
print("앞:", piece[:200])
print("끝:", piece[-200:])
print("\n안에 있는 Section 제목:", re.findall(r"Section \d\s?[—–-]\s?[A-Z][A-Za-z ,]{3,60}", piece))

CD&A : 122,953 ~ 150,910 / 정리 후 26,808자
앞: Compensation Discussion and Analysis Our Compensation Discussion and Analysis (CD&A) includes a detailed discussion of compensation for five named executive officers during the fiscal year ended Decem
끝: intain any benefit plans that cover only named executive officers. We also do not maintain any executive retirement programs such as executive pension plans or supplemental executive retirement plans.

안에 있는 Section 제목: ['Section 1—Executive Summary Compensation Philosophy We designed our emp', 'Section 2—Determining Competitive Levels of Pay Our executive compensat', 'Section 3—Elements of Pay and Fiscal Year ', 'Section 5—Other Compensation Information The first four sections of thi']


In [30]:
pos = piece.find("Section 4")
print(pos, "/", piece[pos:pos+120])

11354 / Section 4—2026 Compensation Decisions 2026 CEO Award The Compensation Committee currently follows a triennial grant cade


## 10. 2번 보충: Audit Matters
감사 보수 표가 든 섹션을 찾아 `proposal_02_supp_audit.txt`로 저장한다.

In [31]:
p2 = agenda.loc[agenda["proposal_no"] == 2, "clean_text"].iloc[0]
print(len(p2), "자\n")
print(p2)

2170 자

Ratification of Appointment of Independent Registered Public Accounting Firm The Audit Committee has appointed Ernst & Young LLP as the independent registered public accounting firm to audit our consolidated financial statements for the fiscal year ending December 31, 2026. During the fiscal year ended December 31, 2025, Ernst & Young LLP served as our independent registered public accounting firm and also provided certain audit-related, tax, and other services. See “Independent Registered Public Accounting Firm” on page 59 of this proxy statement. The Audit Committee believes that the continued retention of Ernst & Young LLP as our independent registered public accounting firm is in the best interests of Alphabet and our shareholders. Notwithstanding its selection, the Audit Committee, in its discretion, may appoint another independent registered public accounting firm at any time during the year if the Audit Committee believes that such a change would be in the best interests

In [32]:
print(text[37800:39000])

ar-End 53 Stock Vested in Fiscal 202 5 54 14 Alphabet 2026 Proxy Statement Potential Payments Upon Termination or Change in Control 55 Alphabet CEO Pay Ratio 56 Alphabet Pay vs. Performance 56 Equity Compensation Plan Information 58 3 Audit Matters Independent Registered Public Accounting Firm 59 Principal Accountant Fees and Services 59 Auditor Independence 59 Pre-Approval Policies and Procedures 59 Report of the Audit Committee of the Board of Directors 60 4 Management and Shareholder Proposals Management Proposals 61 Proposal Number 1: Election of Directors 61 Proposal Number 2: Ratification of Appointment of Independent Registered Public Accounting Firm 62 Proposal Number 3: A pproval of the Amendment and Restatement of Alphabet Inc. Amended and Restated 2021 Stock Plan 63 Proposal Number 4: Advisory Vote to Approve Compensation Awarded to Named Executive Officers 68 Shareholder Proposals 69 Proposal Number 5: Shareholder Proposal Regarding an Enhanced Disclosure on Climate Goals 7

In [33]:
lo, hi = 150910, 194358   # CD&A 끝 ~ 1번 안건 시작

for t in ["Independent Registered Public Accounting Firm", "Management Proposals"]:
    hits = [m.start() for m in title_regex(t).finditer(text, lo, hi)]
    print(f"===== {t} : {len(hits)}회 =====")
    for h in hits:
        print(f"[{h:,}] ...{text[h-60:h+140]}...\n")

===== Independent Registered Public Accounting Firm : 1회 =====
[188,854] ...Executive Compensation Audit Matters Proposals Q&A Appendix Independent Registered Public Accounting Firm Principal Accountant Fees and Services The following table sets forth all fees paid or accrued...

===== Management Proposals : 1회 =====
[194,318] ...Executive Compensation Audit Matters Proposals Q&A Appendix Management Proposals Proposal Number 1: Election of Directors Nominees The Governance Committee recommended, and our Board nominated: • Larr...



In [34]:
start, end = 188854, 194318
piece = clean_section(text[start:end])

out = CLEAN_DIR / "proposal_02_supp_audit.txt"
out.write_text(piece, encoding="utf-8")

print(f"Audit Matters : {start:,} ~ {end:,} / 정리 후 {len(piece):,}자\n")
print("끝:", piece[-200:], "\n")

# 보수 표가 들어 있는지 확인: 'Audit Fees' 주변 400자
fee_pos = piece.find("Audit Fees")
print("Audit Fees 위치:", fee_pos)
print(piece[fee_pos:fee_pos+400])

Audit Matters : 188,854 ~ 194,318 / 정리 후 5,135자

끝: inancial statements be included in Alphabet’s Annual Report on Form 10-K for the year ended December 31, 2025. AUDIT COMMITTEE Roger W. Ferguson Jr., Chair R. Martin “Marty” Chávez Robin L. Washington 

Audit Fees 위치: 294
Audit Fees (1) 31,130 37,900 Audit-Related Fees (2) 11,767 9,506 Tax Fees (3) 1,386 1,894 All Other Fees (4) 21,229 19,400 TOTAL FEES 65,512 68,700 (1) Audit Fees: This category represents fees for professional services provided in connection with the audit of our financial statements, audit of our internal control over financial reporting, review of our quarterly financial statements, and audit s


## 11. 1번 보충: 특수관계 거래
'Certain Relationships and Related Transactions' 섹션을 `proposal_01_supp_related_transactions.txt`로 저장한다.

In [35]:
lo, hi = 88403, 122953   # 겸직 섹션 끝 ~ CD&A 시작

for t in ["Certain Relationships and Related Transactions", "Director Compensation"]:
    hits = [m.start() for m in title_regex(t).finditer(text, lo, hi)]
    print(f"===== {t} : {len(hits)}회 =====")
    for h in hits:
        print(f"[{h:,}] ...{text[h-60:h+140]}...\n")

===== Certain Relationships and Related Transactions : 1회 =====
[108,350] ...Executive Compensation Audit Matters Proposals Q&A Appendix Certain Relationships and Related Transactions Related Party Transactions Policy and Procedure Our written Related Party Transactions Policy...

===== Director Compensation : 2회 =====
[116,490] ...Executive Compensation Audit Matters Proposals Q&A Appendix Director Compensation Board Compensation Arrangements for Non-Employee Directors Alphabet’s director compensation program is designed to att...

[120,752] ...Executive Compensation Audit Matters Proposals Q&A Appendix Director Compensation for 2025 The following table summarizes compensation earned by our directors other than Sundar during 2025. Name Fees ...



In [36]:
start, end = 108350, 116490
piece = clean_section(text[start:end])

out = CLEAN_DIR / "proposal_01_supp_related_transactions.txt"
out.write_text(piece, encoding="utf-8")

print(f"특수관계 거래 : {start:,} ~ {end:,} / 정리 후 {len(piece):,}자\n")
print("앞:", piece[:200], "\n")
print("끝:", piece[-300:])

특수관계 거래 : 108,350 ~ 116,490 / 정리 후 7,811자

앞: Certain Relationships and Related Transactions Related Party Transactions Policy and Procedure Our written Related Party Transactions Policy provides that we will only enter into or ratify a transacti 

끝: ur directors, executive officers, and significant shareholders or their immediate family members. We believe that all such arrangements have been entered into in the ordinary course of business and have been conducted on an arm’s-length basis and do not represent a material interest to such parties.


## 12. 파일 목록(manifest) 만들기
파일별 역할·글자 수를 정리하고, 주주제안은 main 대신 holder+company+overview를 LLM 입력으로 표시한다.

In [37]:
rows = []
for f in sorted(CLEAN_DIR.glob("*.txt")):
    m = re.match(r"proposal_(\d{2})(?:_(.+))?\.txt$", f.name)
    if not m:
        continue                      # overview는 아래에서 따로 처리
    no, suffix = int(m.group(1)), m.group(2)

    if suffix is None:
        role = "main"                 # 안건 전체 본문
    elif suffix in ("holder", "company"):
        role = suffix                 # 주주 요구 / 회사 반론
    else:
        role = suffix                 # supp_directors, supp_cda 등

    rows.append({"proposal_no": no, "role": role, "file": f.name,
                 "chars": len(f.read_text(encoding="utf-8"))})

# 주주제안 공통 섹션은 5~14번 모두에 붙이기
ov = CLEAN_DIR / "shareholder_proposals_overview.txt"
for no in range(5, 15):
    rows.append({"proposal_no": no, "role": "overview", "file": ov.name,
                 "chars": len(ov.read_text(encoding="utf-8"))})

manifest = pd.DataFrame(rows).sort_values(["proposal_no", "role"]).reset_index(drop=True)

# LLM 입력용 표시: 주주제안은 main 대신 holder+company+overview (내용 중복 방지)
is_holder_prop = manifest["proposal_no"] >= 5
manifest["llm_input"] = ~(is_holder_prop & (manifest["role"] == "main"))

MANIFEST_PATH = BASE / "data/processed/googl_2026_manifest.csv"
manifest.to_csv(MANIFEST_PATH, index=False, encoding="utf-8-sig")
print("저장:", MANIFEST_PATH, "/ 행 수:", len(manifest), "\n")

# 안건별 요약: 파일 수, LLM 입력 글자 수
summary = (manifest[manifest["llm_input"]]
           .groupby("proposal_no")
           .agg(files=("file", "count"), llm_chars=("chars", "sum")))
summary

저장: C:\desktop\agent_contest\data\processed\googl_2026_manifest.csv / 행 수: 52 



,files,llm_chars
proposal_no,,
1,7,39723
2,2,7305
3,1,21645
4,2,28975
5,3,15643
6,3,16194
7,3,16129
8,3,17367
9,3,15701


## 13. 청크 나누기
문장 경계에서 500~800자로 자르고, 150자 미만 조각은 앞 조각에 합쳐 `googl_2026_chunks.csv`로 저장한다.

In [38]:
SENT_BOUND = re.compile(r"(?<=[.!?:;])\s+(?=[A-Z“\"(•])|\s+(?=•)")   # 문장/글머리표 경계

def chunk_spans(s, min_len=500, max_len=800):
    """문장 경계에서 자른 (시작, 끝) 위치 목록"""
    bounds = [m.end() for m in SENT_BOUND.finditer(s)] + [len(s)]
    spans, start = [], 0
    for b in bounds:
        if b - start < min_len and b != len(s):
            continue                                   # 아직 짧으면 다음 문장까지 이어 붙임
        while b - start > max_len:                     # 너무 길면 공백 기준 강제 분할
            cut = s.rfind(" ", start + min_len, start + max_len)
            cut = cut if cut != -1 else start + max_len
            spans.append((start, cut))
            start = cut
        spans.append((start, b))
        start = b
    return spans

def chunk_prefix(fname):
    """proposal_05_holder.txt → p05_holder / proposal_03.txt → p03_main"""
    stem = fname.replace(".txt", "")
    if stem.startswith("proposal_"):
        stem = "p" + stem[len("proposal_"):]
        if re.fullmatch(r"p\d{2}", stem):
            stem += "_main"
    return stem.replace("shareholder_proposals_", "")

chunk_rows, check = [], []
for fname in manifest.loc[manifest["llm_input"], "file"].unique():   # overview는 한 번만
    s = (CLEAN_DIR / fname).read_text(encoding="utf-8")
    spans = chunk_spans(s)
    check.append({"file": fname, "covers_all": "".join(s[a:b] for a, b in spans) == s})
    for i, (a, b) in enumerate(spans, 1):
        chunk_rows.append({"chunk_id": f"{chunk_prefix(fname)}_{i:03d}", "file": fname,
                           "start": a, "end": b, "chars": b - a, "text": s[a:b].strip()})

chunks = pd.DataFrame(chunk_rows)
CHUNKS_PATH = BASE / "data/processed/googl_2026_chunks.csv"
chunks.to_csv(CHUNKS_PATH, index=False, encoding="utf-8-sig")

print("저장:", CHUNKS_PATH)
print("조각 수:", len(chunks), "/ 원문 빠짐없이 덮는가:", all(c["covers_all"] for c in check), "\n")
print(chunks["chars"].describe().round(0), "\n")
chunks[["chunk_id", "chars", "text"]].head(5)

저장: C:\desktop\agent_contest\data\processed\googl_2026_chunks.csv
조각 수: 348 / 원문 빠짐없이 덮는가: True 

count    348.0
mean     563.0
std      156.0
min       14.0
25%      524.0
50%      573.0
75%      640.0
max      799.0
Name: chars, dtype: float64 



,chunk_id,chars,text
0,p01_main_001,637,"Election of Directors Nominees The Governance Committee recommended, and our Board nominated: • Larry Page • Sergey Brin • Sundar Pichai • John L...."
1,p01_main_002,737,"In the event that any nominee becomes unavailable or unwilling to serve as a member of our Board, the proxy holders will vote in their discretion ..."
2,p01_main_003,631,"Required Vote Our Bylaws require a majority voting standard for elections of directors. To be elected, a nominee must receive the affirmative FOR ..."
3,p01_main_004,668,"If an incumbent director fails to receive the required number of votes for re-election, the Governance Committee will act on a prompt basis to det..."
4,p01_main_005,147,Alphabet Recommendation OUR BOARD OF DIRECTORS RECOMMENDS A VOTE FOR THE ELECTION TO THE BOARD OF DIRECTORS OF EACH OF THE ABOVEMENTIONED NOMINEES.


In [39]:
short = chunks[chunks["chars"] < 150]
print("150자 미만 조각:", len(short), "\n")
short[["chunk_id", "chars", "text"]]

150자 미만 조각: 17 



,chunk_id,chars,text
4,p01_main_005,147,Alphabet Recommendation OUR BOARD OF DIRECTORS RECOMMENDS A VOTE FOR THE ELECTION TO THE BOARD OF DIRECTORS OF EACH OF THE ABOVEMENTIONED NOMINEES.
51,p01_supp_outside_boards_002,56,All of our directors are in compliance with this policy.
76,p02_supp_audit_002,14,& Young LLP.
84,p02_supp_audit_010,89,"AUDIT COMMITTEE Roger W. Ferguson Jr., Chair R. Martin “Marty” Chávez Robin L. Washington"
118,p03_main_034,17,a single class.
170,p04_supp_cda_046,133,We also do not maintain any executive retirement programs such as executive pension plans or supplemental executive retirement plans.
177,p05_company_007,95,"Unless marked to the contrary, proxies received will be voted AGAINST the shareholder proposal."
229,p07_holder_007,50,"DEF 14A (Apr. 25, 2025); Form 10-K (Feb. 4, 2025)."
242,p08_holder_004,66,viewpoints can be raised internally without fear of retaliation.
252,p09_company_006,95,"Unless marked to the contrary, proxies received will be voted AGAINST the shareholder proposal."


In [40]:
def chunk_spans_merged(s, min_keep=150):
    """chunk_spans 결과에서 min_keep자 미만 조각은 같은 파일의 앞 조각에 합침"""
    merged = []
    for a, b in chunk_spans(s):
        if merged and (b - a) < min_keep:
            merged[-1] = (merged[-1][0], b)      # 앞 조각의 끝을 늘림
        else:
            merged.append((a, b))
    return merged

chunk_rows, check = [], []
for fname in manifest.loc[manifest["llm_input"], "file"].unique():
    s = (CLEAN_DIR / fname).read_text(encoding="utf-8")
    spans = chunk_spans_merged(s)
    check.append("".join(s[a:b] for a, b in spans) == s)
    for i, (a, b) in enumerate(spans, 1):
        chunk_rows.append({"chunk_id": f"{chunk_prefix(fname)}_{i:03d}", "file": fname,
                           "start": a, "end": b, "chars": b - a, "text": s[a:b].strip()})

chunks = pd.DataFrame(chunk_rows)
chunks.to_csv(CHUNKS_PATH, index=False, encoding="utf-8-sig")

print("조각 수:", len(chunks), "/ 원문 빠짐없이 덮는가:", all(check))
print("150자 미만 조각:", (chunks["chars"] < 150).sum(), "\n")
print(chunks["chars"].describe().round(0))

조각 수: 331 / 원문 빠짐없이 덮는가: True
150자 미만 조각: 0 

count    331.0
mean     591.0
std      116.0
min      154.0
25%      532.0
50%      582.0
75%      656.0
max      885.0
Name: chars, dtype: float64


## 14. ESG 라벨 초안
안건 CSV로 라벨 틀을 만들고 Claude 초안(E/S/G 주·부 분류, 메모)을 채운다.

In [41]:
labels = pd.read_csv(AGENDA_PATH)[["proposal_no", "title", "proposer", "board_rec", "page", "filer"]]

for col in ["esg_primary", "esg_secondary", "check_title", "check_rec", "check_page", "label_note"]:
    labels[col] = ""

LABEL_PATH = BASE / "data/processed/googl_labels_2026.csv"
labels.to_csv(LABEL_PATH, index=False, encoding="utf-8-sig")
print("저장:", LABEL_PATH)
labels

저장: C:\desktop\agent_contest\data\processed\googl_labels_2026.csv


,proposal_no,title,proposer,board_rec,page,filer,esg_primary,esg_secondary,check_title,check_rec,check_page,label_note
0,1,Election of Directors,회사,FOR,61,NaN,,,,,,
1,2,Ratification of Appointment of Independent Registered Public Accounting Firm,회사,FOR,62,NaN,,,,,,
2,3,Approval of the Amendment and Restatement of Alphabet Inc. Amended and Restated 2021 Stock Plan,회사,FOR,63,NaN,,,,,,
3,4,Advisory Vote to Approve Compensation Awarded to Named Executive Officers,회사,FOR,68,NaN,,,,,,
4,5,Shareholder Proposal Regarding an Enhanced Disclosure on Climate Goals,주주,AGAINST,72,Trillium ESG Global Equity Fund,,,,,,
5,6,Shareholder Proposal Regarding a Report on Water Usage and AI Development,주주,AGAINST,74,"Inspire Investing, LLC",,,,,,
6,7,Shareholder Proposal Regarding Equal Shareholder Voting,주주,AGAINST,77,"The NorthStar Asset Management, Inc. Funded Pension Plan",,,,,,
7,8,Shareholder Proposal Regarding a Viewpoint Diversity Risk Report,주주,AGAINST,80,The National Center for Public Policy Research,,,,,,
8,9,Shareholder Proposal Regarding a Report on Politicized Content Moderation,주주,AGAINST,83,"Bowyer Research, Inc.",,,,,,
9,10,Shareholder Proposal Regarding a Report on Impact of U.S. Immigration Policy,주주,AGAINST,85,The SOC Investment Group,,,,,,


In [42]:
draft = {
    1: ("G", "", "회사 안건"),
    2: ("G", "", "회사 안건"),
    3: ("G", "", "회사 안건"),
    4: ("G", "", "회사 안건"),
    5: ("E", "", "기후 공시"),
    6: ("E", "", "물 사용 공시, AI는 배경"),
    7: ("G", "", "의결권 구조"),
    8: ("S", "G", "관점 차별 위험 보고서 + 독립위원회 요구"),
    9: ("S", "", "콘텐츠·표현, 기부 파트너십"),
    10: ("S", "", "인력·이민"),
    11: ("S", "", "개인정보, 정부 데이터 접근"),
    12: ("G", "S", "이사회 AI 감독 체계"),
    13: ("S", "", "AI 허위정보의 사회 영향"),
    14: ("S", "G", "AI 데이터 사용 위험 보고, 감독 언급"),
}

labels = pd.read_csv(LABEL_PATH).fillna("")
labels["esg_primary"]   = labels["proposal_no"].map(lambda n: draft[n][0])
labels["esg_secondary"] = labels["proposal_no"].map(lambda n: draft[n][1])
labels["label_note"]    = labels["proposal_no"].map(lambda n: draft[n][2])
labels["label_source"]  = "claude_draft"     # 직접 검토한 줄은 'human'으로 바꾸기

labels.to_csv(LABEL_PATH, index=False, encoding="utf-8-sig")
labels[["proposal_no", "title", "esg_primary", "esg_secondary", "label_note", "label_source"]]

,proposal_no,title,esg_primary,esg_secondary,label_note,label_source
0,1,Election of Directors,G,,회사 안건,claude_draft
1,2,Ratification of Appointment of Independent Registered Public Accounting Firm,G,,회사 안건,claude_draft
2,3,Approval of the Amendment and Restatement of Alphabet Inc. Amended and Restated 2021 Stock Plan,G,,회사 안건,claude_draft
3,4,Advisory Vote to Approve Compensation Awarded to Named Executive Officers,G,,회사 안건,claude_draft
4,5,Shareholder Proposal Regarding an Enhanced Disclosure on Climate Goals,E,,기후 공시,claude_draft
5,6,Shareholder Proposal Regarding a Report on Water Usage and AI Development,E,,"물 사용 공시, AI는 배경",claude_draft
6,7,Shareholder Proposal Regarding Equal Shareholder Voting,G,,의결권 구조,claude_draft
7,8,Shareholder Proposal Regarding a Viewpoint Diversity Risk Report,S,G,관점 차별 위험 보고서 + 독립위원회 요구,claude_draft
8,9,Shareholder Proposal Regarding a Report on Politicized Content Moderation,S,,"콘텐츠·표현, 기부 파트너십",claude_draft
9,10,Shareholder Proposal Regarding a Report on Impact of U.S. Immigration Policy,S,,인력·이민,claude_draft
